# Couple river junction node to nearest river chainage lication

This notebook demonstrates how to batch‑configure river chainage locations for river junction nodes that are missing chainage information.
The tool helps reduce manual configuration work and is especially useful when you need to configure hundreds or even thousands of river junctions.

In [1]:
import mikeplus as mp
from mikeplus.utilities import get_nearest_river_chainage_at
from mikeplus.dotnet import DotNetConverter

db = mp.open("../tests/testdata/RiverCS/River_CS.sqlite")
db

Database<'River_CS.sqlite'>

## Analysis process

The process consists of the following steps:

1) Identify all river‑junction nodes for which the BranchID field is NULL.
2) For each of these nodes, determine the nearest river and calculate the corresponding river chainage.
3) Update the node table by assigning the derived river name and chainage values.

In [2]:
df = (
        db._tables.msm_Node.select(["GeomX", "GeomY"])
        .where("TypeNo=6 AND BranchID is NULL")
        .execute()
    )
db.begin_transaction()
commit = True
try:
    for key, value in df.items():
        x = value[0]
        y = value[1]
        river_chainage = get_nearest_river_chainage_at(
            db, x, y, 100.0
        )
        if river_chainage is not None:
            river = river_chainage[0]
            chainage = river_chainage[1]
            db._tables.msm_Node.update({"BranchID": river, "BranchChainage": chainage}).by_muid(key).execute()
            print(
                f"River junction of '{key}' has been coupled to '{river}' at {chainage}"
            )

except RuntimeError as e:
    print(f"An error occurred: {e}")
    commit = False

finally:
    db.end_transaction(commit)

River junction of 'Node_33' has been coupled to 'River' at 755.0359876508826
River junction of 'Pump_3_Outlet' has been coupled to 'River' at 735.7046165978604


# Close database

In [3]:
# Close the database when you're done using it.
db.close()